In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------
# Config
# ---------------------------------------
FILE_PATH = "session_dataset.csv"
TARGET = "any_addon_added"
CUISINE_COL = "restaurant_cuisine"
MEAL_COL = "meal_time"
MIN_SUPPORT = 5000

# ---------------------------------------
# Load data
# ---------------------------------------
df = pd.read_csv(FILE_PATH)

required_cols = [TARGET, CUISINE_COL, MEAL_COL]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Dataset shape:", df.shape)

# ---------------------------------------
# 1) Overall cuisine conversion with support
# ---------------------------------------
overall = (
    df.groupby(CUISINE_COL)[TARGET]
      .agg(conversion_rate="mean", session_count="count")
      .reset_index()
      .sort_values(["conversion_rate", "session_count"], ascending=[False, False])
      .reset_index(drop=True)
)

print("\n=== Overall cuisine conversion (all cuisines) ===")
print(overall)

reliable_overall = overall[overall["session_count"] >= MIN_SUPPORT].copy()
reliable_overall = reliable_overall.sort_values(
    ["conversion_rate", "session_count"], ascending=[False, False]
).reset_index(drop=True)

if reliable_overall.empty:
    raise ValueError(f"No cuisines have at least {MIN_SUPPORT} sessions.")

best_reliable_cuisine = reliable_overall.iloc[0][CUISINE_COL]
best_reliable_rate = float(reliable_overall.iloc[0]["conversion_rate"])
best_reliable_count = int(reliable_overall.iloc[0]["session_count"])

print(f"\n=== Reliable cuisines only (count >= {MIN_SUPPORT}) ===")
print(reliable_overall)

print("\nBest reliable overall cuisine:", best_reliable_cuisine)
print("Conversion rate:", best_reliable_rate)
print("Session count:", best_reliable_count)

reliable_cuisines = set(reliable_overall[CUISINE_COL])

# Keep only reliable cuisines for the rest of the analysis
df_rel = df[df[CUISINE_COL].isin(reliable_cuisines)].copy()

# ---------------------------------------
# 2) Conditional conversion by meal_time
# ---------------------------------------
conditional = (
    df_rel.groupby([MEAL_COL, CUISINE_COL])[TARGET]
      .agg(conversion_rate="mean", session_count="count")
      .reset_index()
)

print("\n=== Conditional conversion by meal_time and cuisine ===")
print(conditional.sort_values([MEAL_COL, "conversion_rate"], ascending=[True, False]))

best_by_meal = (
    conditional.sort_values(
        [MEAL_COL, "conversion_rate", "session_count"],
        ascending=[True, False, False]
    )
    .groupby(MEAL_COL, as_index=False)
    .first()
)

best_by_meal["matches_overall_best"] = best_by_meal[CUISINE_COL] == best_reliable_cuisine

print("\n=== Best cuisine by meal_time ===")
print(best_by_meal)

all_segments_match = bool(best_by_meal["matches_overall_best"].all())
num_matching_segments = int(best_by_meal["matches_overall_best"].sum())
num_segments = len(best_by_meal)

print("\nOverall best cuisine:", best_reliable_cuisine)
print(f"Matches best cuisine in {num_matching_segments}/{num_segments} meal_time segments")
print("Remains best in all meal segments?", all_segments_match)

# ---------------------------------------
# 3) Meal-time base rates
# ---------------------------------------
meal_base = (
    df_rel.groupby(MEAL_COL)[TARGET]
      .agg(base_conversion_rate="mean", session_count="count")
      .reset_index()
      .sort_values("base_conversion_rate", ascending=False)
)

print("\n=== Meal-time base conversion rates ===")
print(meal_base)

# ---------------------------------------
# 4) Meal-time distribution by cuisine
#    This is important for spurious-correlation analysis
# ---------------------------------------
meal_mix = pd.crosstab(df_rel[CUISINE_COL], df_rel[MEAL_COL], normalize="index")

print("\n=== Meal-time distribution within each cuisine ===")
print(meal_mix.round(4))

# ---------------------------------------
# 5) Standardized conversion rate by cuisine
#    Reweight each cuisine to the GLOBAL meal-time distribution
# ---------------------------------------

# Global meal-time distribution among reliable cuisines
global_meal_weights = df_rel[MEAL_COL].value_counts(normalize=True).sort_index()

# Cuisine x meal_time conversion table
conv_table = conditional.pivot(index=CUISINE_COL, columns=MEAL_COL, values="conversion_rate")

# Ensure same meal_time columns / order
conv_table = conv_table.reindex(columns=global_meal_weights.index)

# If any cuisine/meal slot is missing, leave as NaN and fill with that meal's global base rate
meal_base_map = meal_base.set_index(MEAL_COL)["base_conversion_rate"].to_dict()
for meal in conv_table.columns:
    conv_table[meal] = conv_table[meal].fillna(meal_base_map[meal])

# Standardized rate = sum over meal_time of cuisine-specific rate * global meal weights
standardized = conv_table.mul(global_meal_weights, axis=1).sum(axis=1).reset_index()
standardized.columns = [CUISINE_COL, "standardized_conversion_rate"]

# Merge observed and standardized
spurious_check = reliable_overall.merge(standardized, on=CUISINE_COL, how="left")
spurious_check["difference_observed_minus_standardized"] = (
    spurious_check["conversion_rate"] - spurious_check["standardized_conversion_rate"]
)

spurious_check = spurious_check.sort_values("conversion_rate", ascending=False).reset_index(drop=True)

print("\n=== Observed vs meal-time-standardized cuisine conversion ===")
print(spurious_check)

# ---------------------------------------
# 6) Rank comparison
# ---------------------------------------
observed_rank = (
    spurious_check[[CUISINE_COL, "conversion_rate"]]
    .sort_values("conversion_rate", ascending=False)
    .reset_index(drop=True)
)
observed_rank["observed_rank"] = np.arange(1, len(observed_rank) + 1)

std_rank = (
    spurious_check[[CUISINE_COL, "standardized_conversion_rate"]]
    .sort_values("standardized_conversion_rate", ascending=False)
    .reset_index(drop=True)
)
std_rank["standardized_rank"] = np.arange(1, len(std_rank) + 1)

rank_compare = observed_rank.merge(std_rank, on=CUISINE_COL)
rank_compare["rank_change"] = rank_compare["observed_rank"] - rank_compare["standardized_rank"]

print("\n=== Rank comparison: observed vs standardized ===")
print(rank_compare.sort_values("observed_rank"))

# ---------------------------------------
# 7) Simple heuristic conclusion about spurious correlation
# ---------------------------------------
best_observed = observed_rank.iloc[0][CUISINE_COL]
best_standardized = std_rank.iloc[0][CUISINE_COL]

potential_spurious = (
    (best_observed != best_standardized) or
    (not all_segments_match)
)

print("\n=== Spurious-correlation diagnostic ===")
print("Best cuisine by observed overall rate:", best_observed)
print("Best cuisine by meal-time-standardized rate:", best_standardized)
print("Overall best remains best in all segments?:", all_segments_match)
print("Potential spurious correlation due to meal-time mix?:", potential_spurious)

# ---------------------------------------
# 8) Save outputs
# ---------------------------------------
overall.to_csv("q2_all_cuisines_overall.csv", index=False)
reliable_overall.to_csv("q2_reliable_cuisines_overall.csv", index=False)
conditional.to_csv("q2_conditional_cuisine_mealtime.csv", index=False)
best_by_meal.to_csv("q2_best_cuisine_by_mealtime.csv", index=False)
meal_base.to_csv("q2_mealtime_base_rates.csv", index=False)
meal_mix.to_csv("q2_cuisine_mealtime_mix.csv")
spurious_check.to_csv("q2_observed_vs_standardized.csv", index=False)
rank_compare.to_csv("q2_rank_comparison.csv", index=False)

# ---------------------------------------
# 9) Graphs
# ---------------------------------------

# Reliable observed overall conversion
plt.figure(figsize=(10, 6))
plt.bar(reliable_overall[CUISINE_COL], reliable_overall["conversion_rate"])
plt.title(f"Reliable Cuisine Conversion Rates (min {MIN_SUPPORT} sessions)")
plt.xlabel("Restaurant Cuisine")
plt.ylabel("Observed Conversion Rate")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("q2_reliable_cuisine_conversion.png", dpi=200, bbox_inches="tight")
plt.close()

# Standardized conversion
std_plot = spurious_check.sort_values("standardized_conversion_rate", ascending=False)
plt.figure(figsize=(10, 6))
plt.bar(std_plot[CUISINE_COL], std_plot["standardized_conversion_rate"])
plt.title("Meal-Time-Standardized Cuisine Conversion Rates")
plt.xlabel("Restaurant Cuisine")
plt.ylabel("Standardized Conversion Rate")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("q2_standardized_cuisine_conversion.png", dpi=200, bbox_inches="tight")
plt.close()

# Observed vs standardized comparison
plot_df = spurious_check.set_index(CUISINE_COL)[["conversion_rate", "standardized_conversion_rate"]]
plot_df = plot_df.sort_values("conversion_rate", ascending=False)

x = np.arange(len(plot_df))
width = 0.38

plt.figure(figsize=(12, 6))
plt.bar(x - width/2, plot_df["conversion_rate"], width, label="Observed")
plt.bar(x + width/2, plot_df["standardized_conversion_rate"], width, label="Standardized")
plt.xticks(x, plot_df.index, rotation=45)
plt.ylabel("Conversion Rate")
plt.title("Observed vs Meal-Time-Standardized Cuisine Conversion")
plt.legend()
plt.tight_layout()
plt.savefig("q2_observed_vs_standardized_comparison.png", dpi=200, bbox_inches="tight")
plt.close()

print("\nSaved files:")
print("- q2_all_cuisines_overall.csv")
print("- q2_reliable_cuisines_overall.csv")
print("- q2_conditional_cuisine_mealtime.csv")
print("- q2_best_cuisine_by_mealtime.csv")
print("- q2_mealtime_base_rates.csv")
print("- q2_cuisine_mealtime_mix.csv")
print("- q2_observed_vs_standardized.csv")
print("- q2_rank_comparison.csv")
print("- q2_reliable_cuisine_conversion.png")
print("- q2_standardized_cuisine_conversion.png")
print("- q2_observed_vs_standardized_comparison.png")

# ---------------------------------------
# 10) Reference summary
# ---------------------------------------
summary = {
    "min_support": MIN_SUPPORT,
    "best_reliable_overall_cuisine": best_reliable_cuisine,
    "best_reliable_overall_conversion_rate": best_reliable_rate,
    "best_reliable_overall_session_count": best_reliable_count,
    "best_cuisine_by_meal_time": best_by_meal[[MEAL_COL, CUISINE_COL, "conversion_rate"]].to_dict(orient="records"),
    "overall_best_matches_all_meal_segments": all_segments_match,
    "num_matching_segments": num_matching_segments,
    "num_total_segments": num_segments,
    "best_standardized_cuisine": best_standardized,
    "potential_spurious_correlation_due_to_mealtime_mix": bool(potential_spurious),
}

print("\n=== Reference summary ===")
print(summary)

Dataset shape: (100000, 57)

=== Overall cuisine conversion (all cuisines) ===
  restaurant_cuisine  conversion_rate  session_count
0          fast_food         0.249343           9517
1            chinese         0.238652          13857
2        continental         0.237766           5926
3       south_indian         0.236162          17073
4           desserts         0.235372           6802
5            italian         0.234526          11939
6       north_indian         0.234048          19872
7            biryani         0.230537          11920
8            arabian         0.226568           3094

=== Reliable cuisines only (count >= 5000) ===
  restaurant_cuisine  conversion_rate  session_count
0          fast_food         0.249343           9517
1            chinese         0.238652          13857
2        continental         0.237766           5926
3       south_indian         0.236162          17073
4           desserts         0.235372           6802
5            italian     